# 5.01 BM25 · Ensemble Retriever

키워드 기반 `BM25Retriever`와 이를 **dense 벡터 retriever와 섞는** `EnsembleRetriever`를 다룬다.
- `BM25Retriever` (langchain-community, `rank_bm25` 의존) — 희소 TF-IDF 변형. LLM/임베딩 없이 돌아간다.
- `EnsembleRetriever` (langchain-classic) — 여러 retriever의 랭킹을 **Reciprocal Rank Fusion**으로 병합, 가중치 지정.

## 학습 목표

- `BM25Retriever.from_texts(...)` 팩토리와 `k` 파라미터를 이해한다
- `EnsembleRetriever(retrievers=[...], weights=[...])`에 BM25 + dense를 태워 하이브리드 검색 구성
- 같은 질의를 **순수 BM25 / 순수 dense / 0.3+0.7 혼합**으로 돌려 결과를 비교한다
- `create_retriever_tool(...)`로 ensemble retriever를 에이전트 도구로 노출한다

## 언제 쓰나

- 제품명·모델번호·코드 식별자처럼 **정확 매칭**이 중요한 도메인 (BM25 우위)
- 풀어쓴 자연어 질의에서 **의미 유사도**가 필요한 도메인 (dense 우위)
- 두 특성이 **혼재**하는 사내 문서 · 법령 · 제품 FAQ — ensemble이 두 약점을 상쇄한다


## 5.01.1 환경 설정

`BM25Retriever`는 `rank_bm25` 파이썬 패키지를 요구한다. Ensemble의 dense 측 retriever에는 OpenAI 임베딩을 쓰므로 `OPENAI_API_KEY`가 필요하다.


In [ ]:
# !pip install -U langchain langchain-core langchain-openai langchain-community langchain-classic rank_bm25

import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요 (dense retriever 임베딩용)"

import rank_bm25  # noqa: F401  — import 자체가 설치 검증


## 5.01.2 기본 사용법 — `BM25Retriever`

한국어 + 영문이 섞인 샘플 문서 8개로 시작한다. `from_texts`는 `texts` · `metadatas`를 함께 받는다.
`k`는 반환 개수 기본 4. 인스턴스 속성으로 바꾼다 (`retriever.k = 3`).


In [ ]:
from langchain_community.retrievers import BM25Retriever

corpus = [
    "LangChain은 LLM 애플리케이션 구축을 위한 프레임워크이다.",
    "LangGraph는 상태 기반 멀티 에이전트 워크플로우 라이브러리이다.",
    "FAISS는 Meta가 만든 approximate nearest neighbor 검색 엔진이다.",
    "BM25는 전통적인 lexical TF-IDF 변형 랭킹 함수이다.",
    "RAG는 검색 증강 생성 기법으로 외부 지식을 LLM 응답에 결합한다.",
    "Claude Sonnet 4.6은 Anthropic이 공개한 최신 모델이다.",
    "GPT-4.1과 o4-mini는 OpenAI의 2025년 주력 모델이다.",
    "EnsembleRetriever는 dense와 sparse 랭킹을 RRF로 결합한다.",
]
metas = [{"doc_id": i, "tag": "tech"} for i in range(len(corpus))]

bm25 = BM25Retriever.from_texts(corpus, metadatas=metas)
bm25.k = 3

hits = bm25.invoke("GPT 모델")
for d in hits:
    print("-", d.page_content, "|", d.metadata)


## 5.01.3 Ensemble 구성 — 가중치 · weights

`EnsembleRetriever(retrievers=[r1, r2], weights=[w1, w2])`는 각 retriever가 반환한 순위 리스트에 RRF 점수를 매기고 `weights` 비율로 가중 합산한다.
- 일반 RAG 권장값: **sparse 0.3 / dense 0.7**
- 정확 매칭이 중요한 도메인(코드·식별자): sparse 0.5~0.7
- `c=60` RRF 상수는 내부 기본값, 보통 건드리지 않는다

> NOTE: `weights`는 확률이 아닌 **상대 가중치**. 합이 1이 아니어도 되지만 해석 편의상 맞추는 게 관례.


In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_classic.retrievers.ensemble import EnsembleRetriever

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

dense_store = InMemoryVectorStore.from_texts(corpus, embedding=embeddings, metadatas=metas)
dense = dense_store.as_retriever(search_kwargs={"k": 3})

ensemble = EnsembleRetriever(
    retrievers=[bm25, dense],
    weights=[0.3, 0.7],  # sparse 30% + dense 70%
)

for d in ensemble.invoke("OpenAI의 최신 LLM"):
    print("-", d.page_content)


## 5.01.4 결과 비교 — 같은 질의, 세 설정

`"RRF 랭킹 결합"` 같은 혼합 질의를 세 retriever에 같이 돌려 차이를 본다.
- 순수 BM25는 `RRF`, `랭킹` 같은 **어휘 일치**를 잡는다
- 순수 dense는 "EnsembleRetriever"처럼 **의미적으로 가까운** 문장을 잡는다
- Ensemble(0.3/0.7)은 두 쪽 모두를 위쪽으로 끌어올린다


In [ ]:
query = "RRF 랭킹 결합"

def top_ids(ret, q, k=3):
    return [d.metadata["doc_id"] for d in ret.invoke(q)[:k]]

print(f"질의: {query!r}")
print("순수 BM25        :", top_ids(bm25, query))
print("순수 dense       :", top_ids(dense, query))
print("Ensemble 0.3/0.7 :", top_ids(ensemble, query))

print()
print("--- 참고: 질의를 바꿔 본다 ---")
for q in ["Claude 모델 버전", "벡터 검색 라이브러리"]:
    print(f"{q!r:>22} | BM25={top_ids(bm25,q)}  dense={top_ids(dense,q)}  ens={top_ids(ensemble,q)}")


## 5.01.5 `create_agent` 연동 — retriever를 tool로

`create_retriever_tool(retriever, name, description)`은 `.invoke(query)` 인터페이스를 가진 **문자열 반환 도구**를 만든다.
에이전트는 이 tool을 호출해 문서 덩어리를 받고, 다시 LLM이 답변을 합성한다 — 전형적인 agentic RAG 패턴이다.


In [ ]:
from langchain_classic.tools.retriever import create_retriever_tool
from langchain.agents import create_agent

rag_tool = create_retriever_tool(
    ensemble,
    name="search_tech_docs",
    description="LangChain · LangGraph · LLM 관련 기술 문서를 BM25+dense 하이브리드로 검색한다.",
)

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[rag_tool],
    system_prompt="사용자 질문이 오면 search_tech_docs 도구를 먼저 호출하고, 검색 결과만 근거로 답하라.",
)

result = agent.invoke({"messages": [{"role": "user", "content": "RRF로 결과를 합치는 retriever가 뭐야?"}]})
print(result["messages"][-1].content)


## 체크리스트

- [ ] `BM25Retriever`는 LLM/임베딩 없이 돌지만 **토크나이저가 공백 분리** — 한국어는 형태소 분석기 결합을 고려 (예: konlpy + `preprocess_func`)
- [ ] `weights` 합이 1일 필요는 없지만, 가독성을 위해 정규화해 두는 것이 관례
- [ ] Ensemble은 내부적으로 **각 retriever를 모두 실행** → latency·비용이 합산된다
- [ ] `create_retriever_tool` 반환값은 일반 `StructuredTool` → `create_agent(tools=[...])`에 그대로 투입

## 다음

- `02_multi_vector_and_parent.ipynb` — 요약·가설질문 기반 multi-vector, parent-document 검색
- `05_advanced/05_agentic_rag.ipynb` — retriever 여러 개를 도구로 쓰는 풀 에이전트

## 참고

- LangChain Retrievers: https://python.langchain.com/docs/integrations/retrievers/
- `rank_bm25` 패키지: https://github.com/dorianbrown/rank_bm25
- Reciprocal Rank Fusion (Cormack 2009)
